# Tuần 4 — Vòng lặp Active Learning trên Google Colab

**Trước khi chạy:** Runtime → Change runtime type → **GPU** (T4 hoặc tốt hơn).

## Kết quả tuần 3 đã đạt

| Chỉ số | Kết quả | Tham chiếu wav2vec 2.0 Table 9 |
|---|---|---|
| WER không LM, 1 giờ nhãn | **25,93 %** | 24,5 % |
| CER | 7,32 % | — |

Pipeline đã đúng. Tuần 4 dùng lại chính mô-đun fine-tune đó, chỉ gọi lại nhiều lần.

## Các ô trong notebook này

| Ô | Việc | Thời gian |
|---|---|---|
| 1–3 | GPU, Drive, thư viện | 5 phút |
| 4 | Giải nén LibriSpeech | 6–8 phút |
| 5–6 | Lấy code, dựng manifest | 5 phút |
| 7 | **Kiểm tra logic** (không cần GPU) | 2 giây |
| 8 | Ước lượng ngân sách | tức thì |
| 9 | **Chạy thử đường ống** | 25–40 phút |
| 10 | **Chạy thật, đường cong Random** | 4–6 giờ |
| 11–12 | Sao lưu, vẽ đường cong | 2 phút |

Ô 7 phải xanh trước khi chạy ô 9. Ô 9 phải xong trước khi chạy ô 10.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHONG CO")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

## 2. Mount Google Drive

Drive giữ **tarball dataset** (đã tải từ tuần 3, không tải lại) và **kết quả**.

Không giải nén dataset vào Drive — I/O của Drive rất chậm, đọc 28.500 file flac sẽ mất hàng giờ. Luôn giải nén vào `/content`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/alasr'
os.makedirs(f'{DRIVE}/tarballs', exist_ok=True)
os.makedirs(f'{DRIVE}/runs', exist_ok=True)
print('Drive:', DRIVE)
!ls -lh {DRIVE}/tarballs

## 3. Cài thư viện

**Phải gỡ `peft` trước.** Colab cài sẵn `peft` bản mới, nó yêu cầu `accelerate` cao hơn bản đang khóa, và `transformers.trainer` tự nạp `peft` nếu thấy có → lỗi `cannot import name 'clear_device_cache'`. Dự án không dùng `peft`.

In [ ]:
!pip uninstall -y peft -q
!pip install -q transformers==4.36.2 datasets==2.16.1 accelerate==0.25.0 \
                 soundfile==0.12.1 jiwer==3.0.3 librosa==0.10.1
print("Xong. Neu Colab bao restart runtime -> restart roi chay tiep tu o 4.")

## 4. Giải nén LibriSpeech

Tuần 4 chỉ cần `train-clean-100` (pool) và `test-clean` (đánh giá). `test-other` để dành tuần 10.

In [ ]:
import os

NEEDED = ['train-clean-100.tar.gz', 'test-clean.tar.gz']
URLS = {
    'train-clean-100.tar.gz': 'https://www.openslr.org/resources/12/train-clean-100.tar.gz',
    'test-clean.tar.gz':      'https://www.openslr.org/resources/12/test-clean.tar.gz',
}

for name in NEEDED:
    dst = f'{DRIVE}/tarballs/{name}'
    if not os.path.exists(dst):
        print(f'TAI {name} ...')
        !wget -q --show-progress -O "$dst" "{URLS[name]}"

!mkdir -p /content/data
for name in NEEDED:
    print('Giai nen', name, '...')
    !tar -xzf "{DRIVE}/tarballs/{name}" -C /content/data

!du -sh /content/data/LibriSpeech/train-clean-100 /content/data/LibriSpeech/test-clean
!df -h /content | tail -1

## 5. Lấy code

`data/__init__.py` là file rỗng nhưng **bắt buộc** — không có nó thì `from data.pool import Pool` sẽ lỗi.

In [ ]:
%cd /content
!git clone https://github.com/leminh2212044a/ALASR 2>/dev/null || (cd /content/ALASR && git pull)
%cd /content/ALASR

!touch data/__init__.py models/__init__.py scripts/__init__.py eval/__init__.py

print('\n--- Cau truc ---')
!find . -name __pycache__ -prune -o -name '.git' -prune -o -type f -print | sort

**Phải thấy đủ 6 file của tuần 4:**

```
./run_al.py
./data/pool.py
./data/oracle.py
./strategies/base.py
./strategies/random_strategy.py
./scripts/test_week4.py
```

Thiếu file nào thì chưa push lên GitHub, hoặc `git pull` chưa chạy.

## 6. Dựng manifest

In [ ]:
%cd /content/ALASR
for split in ['train-clean-100', 'test-clean']:
    !python scripts/prepare_manifest.py \
        --librispeech-root /content/data/LibriSpeech \
        --split {split} --out-dir data/librispeech

!ls -lh data/librispeech/

**Kiểm tra bắt buộc** cho `train-clean-100`:

| Chỉ số | Giá trị đúng |
|---|---|
| Số utterance | **28.539** |
| Tổng thời lượng | **~100,6 giờ** |
| Số speaker | **251** |

Lệch nhiều → giải nén thiếu hoặc sai đường dẫn.

## 7. ⭐ Kiểm tra logic tuần 4 — không cần GPU

Chín kiểm tra, chạy khoảng 2 giây trên dữ liệu giả. **Chạy ô này trước, đừng đốt GPU khi logic còn sai.**

Nội dung kiểm tra:

1. Ngân sách tính bằng giờ, không bằng số câu
2. Lấy theo ngân sách không bao giờ vượt quá
3. Chiến lược giữ đúng thứ tự ưu tiên
4. Oracle là đường duy nhất tới phiên âm
5. Oracle bắt được truy vấn trùng
6. `audit()` phát hiện rò rỉ nhãn
7. Pool từ chối manifest lẫn phiên âm
8. Chọn ngẫu nhiên tái lập được, mỗi vòng cho thứ tự khác
9. Lưu và nạp lại trạng thái

In [ ]:
!python scripts/test_week4.py

## 8. Ước lượng ngân sách trước khi chạy

Tốc độ **đo được thật** từ lần chạy thử: **2,03 s/bước** trên T4 (batch 4×4, câu trung bình 12,7 giây). Tắt gradient checkpointing giúp giảm từ 2,32 xuống 2,03 — khoảng 13%, không phải 40% như dự đoán ban đầu; phần lớn thời gian là tính toán thật chứ không phải tính lại.

Thời gian giải mã: khoảng 0,10 s/câu, không đáng kể.

**Số bước tối thiểu để mô hình thoát blank collapse**

Đọc lại log neo tuần 3: loss đứng ở mức 2,85 cho tới khoảng **bước 900–1000** rồi mới sụt xuống. Trước mốc đó mô hình chỉ đoán ký tự rỗng, WER bằng 100%.

Hệ quả: `max_steps` dưới 1500 cho WER 100% ở mọi vòng — đường cong phẳng, vô nghĩa để so sánh chiến lược. **Không được giảm số bước để tiết kiệm thời gian.** Phải giảm số vòng.

In [ ]:
SEC_PER_STEP = 2.03     # do duoc thuc te tren T4
SEC_PER_UTT  = 0.10     # toc do decode

def uoc_luong(rounds, max_steps, n_test):
    n_runs = rounds + 1                       # them vong 0 (seed set)
    train_h = n_runs * max_steps * SEC_PER_STEP / 3600
    eval_h  = n_runs * n_test * SEC_PER_UTT / 3600
    return train_h + eval_h

print(f"{'cau hinh':<40} {'GPU-hour':>9}  {'ghi chu':<22}")
print("-" * 75)
for r, s, n, note in [
    (4, 3000, 1000, "chat luong tot nhat"),
    (4, 2000, 1000, "can 2 phien Colab"),
    (3, 2000, 1000, "can 2 phien Colab"),
    (2, 2000, 1000, "VUA 1 phien <- bat dau tu day"),
    (1, 2000, 1000, "chi 2 diem, qua it"),
    (4, 1000, 1000, "WER 100% moi vong, VO NGHIA"),
]:
    print(f"{r+1} vong x {s} buoc, test {n:>4} cau{'':<10} "
          f"{uoc_luong(r, s, n):>7.1f}  {note:<22}")
print("-" * 75)
print("Mot phien Colab mien phi: khoang 3-4 gio.")
print()
print("KE HOACH: chay --rounds 2 (1,2,3 gio) trong phien nay,")
print("          phien sau chay --rounds 4 --resume de them 4 va 5 gio.")

**Vì sao cố định số bước cho mọi vòng**

Nếu để số bước tăng theo số giờ dữ liệu, vòng sau được **cả nhiều dữ liệu hơn lẫn nhiều bước cập nhật hơn** — hai yếu tố trộn vào nhau, không kết luận được cái nào gây ra chênh lệch. Cố định số bước thì khác biệt duy nhất giữa các vòng là *dữ liệu nào được chọn*.

Đổi lại, 2000 bước cho WER cao hơn 3000 bước vài điểm. Không sao: mọi chiến lược chịu cùng một số bước, nên so sánh *giữa các chiến lược* vẫn công bằng. Chỉ cần ghi rõ trong báo cáo rằng con số tuyệt đối không so trực tiếp với neo 25,93 % ở 3000 bước.

## 9. ⭐ Chạy thử đường ống

Ba vòng, mỗi vòng 400 bước, đánh giá trên 300 câu. Khoảng 25–40 phút.

**Ở ô này không xem WER** — 400 bước quá ít, mô hình chắc chắn chưa học được gì.

In [ ]:
%cd /content/ALASR
!python run_al.py \
    --train-manifest    data/librispeech/train-clean-100.manifest.jsonl \
    --train-transcripts data/librispeech/train-clean-100.transcripts.jsonl \
    --test-manifest     data/librispeech/test-clean.manifest.jsonl \
    --test-transcripts  data/librispeech/test-clean.transcripts.jsonl \
    --strategy random --rounds 2 --max-steps 400 \
    --test-subset 300 --seed 0 --keep-last-only \
    --out-dir runs/smoke_al

**Những gì phải thấy**

| Dấu hiệu | Ý nghĩa |
|---|---|
| `Oracle: nap 28539 transcript` | phiên âm nạp đúng một chỗ duy nhất |
| `audit OK: ... khong ro ri nhan` ở cả 3 vòng | không có câu nào vào tập có nhãn mà chưa qua Oracle |
| `Pool: ... 1.00 gio` → `2.00 gio` → `3.00 gio` | ngân sách cộng đúng |
| `random chon ... cau = 1.000 gio` | lấy theo ngân sách hoạt động |
| `mo dong Transformer` ở 15 % số bước mỗi vòng | giai đoạn chỉ huấn luyện lớp ra hoạt động |
| **không** có cảnh báo `use_reentrant` | gradient checkpointing đã tắt thật |
| bảng đường cong học in ra ở cuối, 3 dòng | ghi kết quả hoạt động |

Nếu còn cảnh báo `use_reentrant` thì bản `w2v2_ctc.py` trên GitHub chưa được cập nhật — pull lại.

**Đo lại tốc độ thực tế** từ log của ô này (dòng `1.xx s/it`), rồi sửa `SEC_PER_STEP` ở ô 8 và chạy lại ô 8 để chốt cấu hình.

## 10. ⭐ Chạy thật — đường cong Random

Chia thành hai phiên. Mỗi phiên khoảng 3,5 giờ, vừa một phiên Colab miễn phí.

**Phiên này:** 3 điểm đầu — 1, 2, 3 giờ.

Chạy ô 13 (chống mất session) ở một tab khác trước khi bắt đầu.

In [ ]:
%cd /content/ALASR
!python -u run_al.py \
    --train-manifest    data/librispeech/train-clean-100.manifest.jsonl \
    --train-transcripts data/librispeech/train-clean-100.transcripts.jsonl \
    --test-manifest     data/librispeech/test-clean.manifest.jsonl \
    --test-transcripts  data/librispeech/test-clean.transcripts.jsonl \
    --strategy random --rounds 2 --max-steps 2000 \
    --test-subset 1000 --seed 0 --keep-last-only \
    --out-dir runs/al_random_s0 2>&1 | tee -a runs/al_random_s0.log

### Phiên sau — chạy tiếp hai điểm 4 và 5 giờ

Trạng thái pool được lưu sau **mỗi** vòng, nên không mất phần đã chạy. Chạy lại ô 2 đến ô 6 để dựng lại môi trường, rồi chạy ô này:

```python
%cd /content/ALASR
# nap lai ket qua phien truoc tu Drive
!cp -r /content/drive/MyDrive/alasr/runs/al_random_s0 runs/

!python -u run_al.py \
    --train-manifest    data/librispeech/train-clean-100.manifest.jsonl \
    --train-transcripts data/librispeech/train-clean-100.transcripts.jsonl \
    --test-manifest     data/librispeech/test-clean.manifest.jsonl \
    --test-transcripts  data/librispeech/test-clean.transcripts.jsonl \
    --strategy random --rounds 4 --max-steps 2000 \
    --test-subset 1000 --seed 0 --keep-last-only --resume \
    --out-dir runs/al_random_s0 2>&1 | tee -a runs/al_random_s0.log
```

`--resume` tìm file trạng thái mới nhất trong `--out-dir`, phục hồi Oracle, kiểm tra lại không rò rỉ nhãn, rồi bắt đầu từ vòng tiếp theo. Log gán nhãn được ghi tiếp chứ không xoá.

Nếu `--resume` mà chưa có trạng thái nào thì nó chạy từ vòng 0 như bình thường, nên để cờ này luôn cũng không sao.

**Bắt buộc:** sau mỗi phiên phải chạy ô 11 để sao lưu sang Drive, nếu không phiên sau không có gì để nối tiếp.

## 11. Sao lưu và xem kết quả

In [ ]:
import json, shutil, os

SRC = 'runs/al_random_s0'
shutil.copytree(SRC, f'{DRIVE}/runs/al_random_s0', dirs_exist_ok=True)
print('Da sao luu sang Drive')

rows = [json.loads(l) for l in open(f'{SRC}/results.jsonl')]
print()
print(f"{'vong':>4} | {'gio nhan':>9} | {'so cau':>7} | {'WER %':>7} | {'CER %':>7} | {'phut':>6}")
print("-" * 62)
for r in rows:
    print(f"{r['round']:>4} | {r['labeled_hours']:>9.2f} | {r['n_labeled']:>7} | "
          f"{r['wer']:>7.2f} | {r['cer']:>7.2f} | {r['round_sec']/60:>6.0f}")
print("-" * 62)

# Kiem tra tinh hop le: WER phai GIAM khi them du lieu
wers = [r['wer'] for r in rows]
if all(wers[i] >= wers[i+1] for i in range(len(wers)-1)):
    print("WER giam don dieu -> duong ong dung.")
else:
    print("CANH BAO: WER KHONG giam don dieu.")
    print("  - Chenh lech nho (duoi 1 diem) o muc it nhan: co the la nhieu, chap nhan duoc.")
    print("  - Chenh lech lon: co loi. Kiem tra so buoc, learning rate, va so cau rong.")

n_oracle = sum(1 for _ in open(f'{SRC}/oracle_log.jsonl'))
print(f"\nNhat ky gan nhan: {n_oracle} dong = {rows[-1]['n_labeled']} cau co nhan "
      f"-> {'KHOP' if n_oracle == rows[-1]['n_labeled'] else 'KHONG KHOP, kiem tra lai'}")

## 12. Vẽ đường cong học

In [ ]:
import json
import matplotlib.pyplot as plt

rows = [json.loads(l) for l in open('runs/al_random_s0/results.jsonl')]
x = [r['labeled_hours'] for r in rows]
y = [r['wer'] for r in rows]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(x, y, 'o-', lw=2, ms=7, label='Random')
for xi, yi in zip(x, y):
    ax.annotate(f'{yi:.1f}', (xi, yi), textcoords='offset points',
                xytext=(0, 9), ha='center', fontsize=9)

ax.set_xlabel('So gio du lieu co nhan')
ax.set_ylabel('WER (%) tren test-clean, khong LM')
ax.set_title('Duong cong hoc — chien luoc chon ngau nhien (seed 0)')
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig('runs/al_random_s0/learning_curve.png', dpi=150)
fig.savefig(f'{DRIVE}/runs/al_random_s0/learning_curve.png', dpi=150)
plt.show()
print('Da luu learning_curve.png vao Drive')

## 13. Chống mất session

Colab ngắt kết nối sau khoảng 90 phút không tương tác. Chạy ô này **ở một tab riêng** để giữ session trong lúc ô 10 chạy.

In [ ]:
import IPython
IPython.display.display(IPython.display.Javascript('''
  function keepAlive(){
    document.querySelector("colab-toolbar-button#connect")?.click();
    console.log("keep-alive", new Date().toLocaleTimeString());
  }
  setInterval(keepAlive, 60000);
'''))
print("Keep-alive da bat.")

## Checklist cuối tuần 4

- [ ] `test_week4.py` xanh cả 9 mục
- [ ] Chạy thử: 3 vòng đi hết, `audit OK` ở cả 3 vòng, số giờ tăng 1 → 2 → 3
- [ ] Không còn cảnh báo `use_reentrant` trong log
- [ ] `results.jsonl` có 5 dòng, số giờ xấp xỉ 1, 2, 3, 4, 5
- [ ] WER giảm theo số giờ
- [ ] Số dòng nhật ký gán nhãn khớp số câu có nhãn
- [ ] `learning_curve.png` đã có trên Drive
- [ ] Ghi lại tốc độ thực tế (s/bước) để tính ngân sách tuần 5

## Xử lý sự cố

| Lỗi | Nguyên nhân | Cách sửa |
|---|---|---|
| `ModuleNotFoundError: No module named 'data'` | thiếu `data/__init__.py` | chạy lại ô 5 |
| `cannot import name 'clear_device_cache'` | `peft` của Colab | `!pip uninstall -y peft -q` rồi restart runtime |
| `FileNotFoundError: ...manifest.jsonl` | chưa dựng manifest, hoặc sai thư mục hiện tại | `%cd /content/ALASR` rồi chạy lại ô 6 |
| `KeyError: ... khong con trong tap chua nhan` | chiến lược trả về câu đã có nhãn | lỗi trong chiến lược, không phải lỗi vòng lặp |
| `AssertionError: KHONG KHOP: Oracle da query ...` | có chỗ đọc phiên âm không qua Oracle | đây là rò rỉ nhãn, phải sửa ngay, kết quả hiện tại không dùng được |
| `CUDA out of memory` | batch quá lớn cho T4 | giảm `PER_DEVICE_BATCH` xuống 2 và tăng `GRAD_ACCUM` lên 8 trong `models/w2v2_ctc.py` |
| WER của mọi vòng đều 100 % | mô hình chỉ đoán ký tự rỗng | số bước quá ít; kiểm tra log có `mo dong Transformer` chưa |

## Tiếp theo — tuần 5

Thêm chiến lược dựa trên độ không chắc chắn. Việc đầu tiên không phải viết chiến lược, mà là **đo thiên lệch độ dài**: dùng lại mô hình vòng 0 của tuần này, tính điểm cho 2000 câu, đo tương quan giữa điểm và thời lượng câu. Nếu tương quan cao thì điểm số đang đo độ dài chứ không đo độ khó, và mọi thí nghiệm sau đó vô nghĩa.